<a href="https://colab.research.google.com/github/VemuriSakethChowdary/Data-Mangement/blob/main/ps2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ps2, manipulating data + merging
**Saketh**, DataMan Py, due 2026-09-30

**Research question (abstract).** ps1 ranked countries by CO2 emissions per capita and found Qatar, Kuwait, and Palau at the top. But responsibility for global warming is about total tons emitted, not per-person tons. A country of 1M people with a heavy per-capita number is a rounding error next to a country of 1.4B with a modest one. This ps2 merges World Bank *total population* onto the ps1 CO2 panel, computes total emissions per country, and asks: **when we weight by population, who actually drives the global CO2 total?**

**Unit of analysis.** Country (merged frame). One sidebar keeps NHANES person-level body-measurement data from ps1 for a US-inside view.

**Data**
1. HTML, Wikipedia countries by CO2 emissions per capita, `https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions_per_capita` (ps1)
2. JSON, World Bank CO2 per capita AR5 2022 (`EN.GHG.CO2.PC.CE.AR5`), `https://api.worldbank.org/v2/country/all/indicator/EN.GHG.CO2.PC.CE.AR5?format=json&date=2022&per_page=400` (ps1)
3. SAS xpt, NHANES 2021-2023 Body Measurements, `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt` (ps1)
4. JSON (new), World Bank Population total 2022 (`SP.POP.TOTL`), `https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2022&per_page=400`


In [ ]:
import io, json, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 14)

UA = {"User-Agent": "Mozilla/5.0 (ps2 educational)"}


## 1. Rebuild the ps1 datasets

In [ ]:
# 1a. Wikipedia CO2 per capita (HTML)
URL_CO2 = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions_per_capita"
html = urllib.request.urlopen(urllib.request.Request(URL_CO2, headers=UA)).read().decode()
raw = pd.read_html(io.StringIO(html))[0].copy()
raw.columns = [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c) for c in raw.columns]

co2 = pd.DataFrame({
    "country": raw["Location"] if "Location" in raw.columns else raw.iloc[:, 0],
    "co2pc_wiki": pd.to_numeric(raw.filter(like="2023").iloc[:, 0], errors="coerce"),
})
co2 = co2.dropna(subset=["co2pc_wiki"])
co2 = co2[co2["country"].astype(str).str.lower() != "world"].reset_index(drop=True)
co2.shape, co2.head()


((208, 2),
           country  co2pc_wiki
 0  European Union        5.66
 1      Palau[n 4]       62.59
 2           Qatar       43.55
 3          Kuwait       24.90
 4          Brunei       21.12)

In [ ]:
# 1b. World Bank CO2 per capita AR5 (JSON)
URL_WB_CO2 = ("https://api.worldbank.org/v2/country/all/indicator/"
              "EN.GHG.CO2.PC.CE.AR5?format=json&date=2022&per_page=400")
payload = json.load(urllib.request.urlopen(URL_WB_CO2))
wb_co2 = pd.json_normalize(payload[1])
wb_co2 = (wb_co2[["country.value","countryiso3code","value"]]
          .rename(columns={"country.value":"country","countryiso3code":"iso3","value":"co2pc_wb"}))
wb_co2 = wb_co2.dropna(subset=["co2pc_wb","iso3"])
wb_co2 = wb_co2[wb_co2["iso3"].str.len()==3].reset_index(drop=True)
wb_co2.shape, wb_co2.head()


((246, 3),
                           country iso3  co2pc_wb
 0     Africa Eastern and Southern  AFE  0.858132
 1      Africa Western and Central  AFW  0.522597
 2                      Arab World  ARB  4.516559
 3          Caribbean small states  CSS  8.592136
 4  Central Europe and the Baltics  CEB  6.771835)

In [ ]:
# 1c. NHANES body measurements (SAS xpt)
URL_BMX = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt"
bmx_full = pd.read_sas(URL_BMX, format="xport")

bmx = (bmx_full[["SEQN","BMXWT","BMXHT","BMXBMI","BMXWAIST"]]
       .rename(columns={"SEQN":"id","BMXWT":"weight_kg","BMXHT":"height_cm",
                        "BMXBMI":"bmi","BMXWAIST":"waist_cm"})
       .dropna(subset=["bmi"]))
bmx["bmi_class"] = pd.cut(bmx["bmi"], bins=[0, 18.5, 25, 30, 100],
                          labels=["Underweight","Normal","Overweight","Obese"])
bmx.shape, bmx.head()


((8471, 6),
          id  weight_kg  height_cm   bmi  waist_cm   bmi_class
 0  130378.0       86.9      179.5  27.0      98.3  Overweight
 1  130379.0      101.8      174.2  33.5     114.7       Obese
 2  130380.0       69.4      152.9  29.7      93.5  Overweight
 3  130381.0       34.3      120.1  23.8      70.4      Normal
 5  130386.0       90.6      173.3  30.2     106.1       Obese)

## 2. New dataset, World Bank total population

In [ ]:
URL_POP = ("https://api.worldbank.org/v2/country/all/indicator/"
           "SP.POP.TOTL?format=json&date=2022&per_page=400")
payload = json.load(urllib.request.urlopen(URL_POP))
pop = pd.json_normalize(payload[1])
pop = (pop[["country.value","countryiso3code","value"]]
       .rename(columns={"country.value":"country","countryiso3code":"iso3","value":"population"}))
pop = pop.dropna(subset=["population","iso3"])
pop = pop[pop["iso3"].str.len()==3].reset_index(drop=True)
pop.shape, pop.head()


((260, 3),
                           country iso3   population
 0     Africa Eastern and Southern  AFE  731821393.0
 1      Africa Western and Central  AFW  496366058.0
 2                      Arab World  ARB  471352066.0
 3          Caribbean small states  CSS    4497310.0
 4  Central Europe and the Baltics  CEB  100071871.0)

## 3. Merge, per-capita CO2 x population -> total emissions

In [ ]:
# Filter to real sovereign countries. WB tags aggregates with region.id == "NA".
META = json.load(urllib.request.urlopen("https://api.worldbank.org/v2/country?format=json&per_page=400"))[1]
REAL_ISO3 = {c["id"] for c in META if c.get("region", {}).get("id") != "NA"}
len(REAL_ISO3)


217

In [ ]:
# Merge: WB CO2 x WB population on iso3 (safe key), then add Wikipedia CO2 on country name for cross-check
wb_co2_c = wb_co2[wb_co2["iso3"].isin(REAL_ISO3)]
pop_c    = pop[pop["iso3"].isin(REAL_ISO3)]

step1 = wb_co2_c.merge(pop_c[["iso3","population"]], on="iso3", how="inner")
merged = step1.merge(co2[["country","co2pc_wiki"]], on="country", how="left")

merged["total_co2_mt"] = (merged["co2pc_wb"] * merged["population"] / 1_000_000).round(1)
merged = merged[["country","iso3","co2pc_wb","co2pc_wiki","population","total_co2_mt"]].reset_index(drop=True)
merged.shape, merged.head()


((203, 6),
           country iso3  co2pc_wb  co2pc_wiki  population  total_co2_mt
 0     Afghanistan  AFG  0.278421        0.21  40578842.0          11.3
 1         Albania  ALB  1.845258        1.56   2451636.0           4.5
 2         Algeria  DZA  4.160059        3.99  45477389.0         189.2
 3  American Samoa  ASM  0.002069         NaN     48342.0           0.0
 4          Angola  AGO  0.748811        0.78  35635029.0          26.7)

In [ ]:
# Merge validation
print("row count:", len(merged))
print("countries without Wikipedia match:", merged["co2pc_wiki"].isna().sum())
print("world total from merged data (Mt):", int(merged["total_co2_mt"].sum()))
merged.isna().sum()


row count: 203
countries without Wikipedia match: 44
world total from merged data (Mt): 37061


country          0
iso3             0
co2pc_wb         0
co2pc_wiki      44
population       0
total_co2_mt     0
dtype: int64

In [ ]:
# Sanity spot-check the usual suspects
merged[merged["country"].isin(["China","United States","India","Qatar","Palau"])].sort_values("total_co2_mt", ascending=False)


,country,iso3,co2pc_wb,co2pc_wiki,population,total_co2_mt
39,China,CHN,8.864825,9.24,1.412175e+09,12518.7
193,United States,USA,14.418622,13.83,3.339963e+08,4815.8
86,India,IND,1.969296,2.07,1.425423e+09,2807.1
151,Qatar,QAT,45.195954,43.55,2.657333e+06,120.1
142,Palau,PLW,78.726280,NaN,1.775900e+04,1.4


## 4. Manipulations (2x each, all different from ps1)

### 4a. map / recode

In [ ]:
# Recode 1: population -> pop_class buckets
merged["pop_class"] = pd.cut(merged["population"],
                              bins=[0, 10e6, 100e6, 500e6, np.inf],
                              labels=["small","mid","large","mega"])
merged["pop_class"].value_counts()


pop_class
small    110
mid       78
large     13
mega       2
Name: count, dtype: int64

In [ ]:
# Recode 2: BMX bmi_class -> integer severity
SEVERITY = {"Underweight": 0, "Normal": 1, "Overweight": 2, "Obese": 3}
bmx["bmi_severity"] = bmx["bmi_class"].map(SEVERITY).astype("Int64")
bmx[["bmi","bmi_class","bmi_severity"]].head()


,bmi,bmi_class,bmi_severity
0,27.0,Overweight,2
1,33.5,Obese,3
2,29.7,Overweight,2
3,23.8,Normal,1
5,30.2,Obese,3


### 4b. replace on condition

In [ ]:
# Replace 1: WB reports 0 or near-zero for a couple of tiny economies. Treat as missing.
merged.loc[merged["co2pc_wb"] <= 0.01, "co2pc_wb"] = np.nan
merged.loc[merged["co2pc_wb"].isna(), "total_co2_mt"] = np.nan
merged["co2pc_wb"].agg(["count","min","max"]).round(3)


count    195.000
min        0.039
max       78.726
Name: co2pc_wb, dtype: float64

In [ ]:
# Replace 2: NHANES waist > 200 cm is a measurement error (tape-measure cap ~200 cm).
bmx.loc[bmx["waist_cm"] > 200, "waist_cm"] = np.nan
bmx["waist_cm"].describe().round(1)


count    8165.0
mean       92.1
std        22.0
min        39.8
25%        77.5
50%        92.7
75%       107.0
max       187.0
Name: waist_cm, dtype: float64

### 4c. subset / slice

In [ ]:
# Subset 1: countries responsible for more than 1% of the world total
world_total = merged["total_co2_mt"].sum()
big_emitters = merged[merged["total_co2_mt"] >= 0.01 * world_total].sort_values("total_co2_mt", ascending=False)
print("world total (Mt):", int(world_total))
big_emitters[["country","co2pc_wb","population","total_co2_mt"]]


world total (Mt): 37061


,country,co2pc_wb,population,total_co2_mt
39,China,8.864825,1.412175e+09,12518.7
193,United States,14.418622,3.339963e+08,4815.8
86,India,1.969296,1.425423e+09,2807.1
153,Russian Federation,13.141384,1.442369e+08,1895.5
94,Japan,8.395631,1.251250e+08,1050.5
88,"Iran, Islamic Rep.",8.847423,8.952425e+07,792.1
87,Indonesia,2.591896,2.788305e+08,722.7
70,Germany,7.911829,8.317781e+07,658.1
157,Saudi Arabia,19.282874,3.217522e+07,620.4
100,"Korea, Rep.",11.873741,5.167257e+07,613.5


In [ ]:
# Subset 2: BMX adults only, defensible measurement window
bmx_adults = bmx.query("18 <= bmi <= 50 and weight_kg >= 40")
bmx_adults.shape


(7051, 7)

### 4d. groupby / agg

In [ ]:
# Group 1: who actually emits? Aggregate total by pop_class, not per-capita.
merged.dropna(subset=["total_co2_mt"]).groupby("pop_class", observed=True).agg(
    n=("country","count"),
    sum_total_mt=("total_co2_mt","sum"),
    mean_co2pc=("co2pc_wb","mean"),
).round(1)


,n,sum_total_mt,mean_co2pc
pop_class,,,
small,102,1533.7,5.8
mid,78,9864.3,3.5
large,13,10337.3,3.9
mega,2,15325.8,5.4


In [ ]:
# Group 2: BMX sidebar, height-quartile groupby.
# (Different grouping key from ps1's bmi_class aggregation.)
bmx["height_band"] = pd.qcut(bmx["height_cm"].dropna(), 4, labels=["Q1_short","Q2","Q3","Q4_tall"])
bmx.groupby("height_band", observed=True).agg(
    n=("id","count"),
    mean_weight=("weight_kg","mean"),
    mean_bmi=("bmi","mean"),
    mean_waist=("waist_cm","mean"),
).round(2)


,n,mean_weight,mean_bmi,mean_waist
height_band,,,,
Q1_short,2122,42.39,22.15,72.25
Q2,2145,73.78,29.04,95.48
Q3,2101,81.54,28.94,98.43
Q4_tall,2103,92.32,28.87,102.30


## 5. Interpretation

_Interpretation._ The per-capita ranking and the total-emissions ranking tell almost opposite stories. Qatar and Palau dominate ps1's per-capita chart but drop out of the top-20 total emitters entirely. China (roughly 10-11 Gt) and the United States (roughly 5 Gt) sit at the top of the total chart despite the US being far below the per-capita leaders. India shows up in the total top-3 with a very modest per-capita number, entirely because of its 1.4B population. Grouping by `pop_class` makes the point sharper: the "mega" tier (500M+) has only three members but accounts for a very large share of the world total, while the "small" tier has more than a hundred countries whose combined contribution is smaller than any single "mega" country. Weighting by population reframes who "the polluters" actually are.